# __Sepsis Cases Case Study:__ _DyLoPro: Profiling the Dynamics of Event Logs_

The Sepsis Cases event log records the pathways of patients with a suspicion of sepsis through a hospital, from their registration at the emergency room (ER) until their release, and possibly their return to the ER. It consists of 1,050 cases and 15,214 events, recorded between November 2013 and June 2015. The event log can be downloaded [here](https://doi.org/10.4121/uuid:915d2bfb-7e84-49ad-a286-dc35f063a460).

This notebook contains the __case study on the dynamics over time in the Sepsis Cases event log__, following the same structure as the BPIC17 and BPIC19 case studies. Each section corresponds to a certain event log conceptualization, and contains the most interesting identified trends, patterns or changes over time. The visualizations are accompanied by markdown cells discussing these dynamics.

0. [__General Remarks and Observations__](#0-general-observations-and-remarks)
1. [__Variants__](#1-variants)
1. [__Directly-Follows-Relations__](#2-directly-follows-relations-dfrs)
1. [__Categorical Case Features__](#3-categorical-case-features)
1. [__Numerical Case Features__](#4-numerical-case-features)
1. [__Categorical Event Features__](#5-categorical-event-features)
1. [__Numerical Event Features__](#6-numerical-event-features)

Importing the libraries and recording the environment and library versions.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import sys, platform, json, hashlib, datetime as dt
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import pm4py
import DyLoPro as dlp

ENV = {
    'run date': dt.datetime.now().strftime('%Y-%m-%d %H:%M'),
    'python': sys.version.split()[0],
    'platform': platform.platform(),
    'processor': platform.processor() or platform.machine(),
    'DyLoPro': version('DyLoPro'),
    'pandas': pd.__version__,
    'numpy': np.__version__,
    'matplotlib': matplotlib.__version__,
    'pm4py': version('pm4py'),
}
assert int(pd.__version__.split('.')[0]) < 3, 'DyLoPro 0.1.1 does not work with pandas 3: pip install "pandas>=2.0,<3.0"'
pd.DataFrame.from_dict(ENV, orient = 'index', columns = ['value'])

Configuration of the data, the common plot parameters and the output folders. `DOWNLOAD_DATE` is the date the log was downloaded from 4TU.ResearchData.

In [ ]:
DATA_PATH = Path('Sepsis Cases - Event Log.xes/Sepsis Cases - Event Log.xes')
DATA_DOI = 'https://doi.org/10.4121/uuid:915d2bfb-7e84-49ad-a286-dc35f063a460'
DOWNLOAD_DATE = '2026-10-14'          # file date of the downloaded .xes; replace if the exact download date differs

ORIGINAL_CODE = {
    'DyLoPro library (github.com/BrechtWts/DyLoPro)': '2d88d48db881af192c78b00fc12afec919a026d1 (2024-07-23); PyPI release 0.1.1',
    'Case-study notebooks (github.com/BrechtWts/DyLoPro_CaseStudies)': '1b5fe667aab0aafae05a1fe86fc3ce78f0ea3b6b (2023-07-28)',
}

PARAMETERS = dict(time_unit = 'days', frequency = 'monthly', case_assignment = 'first_event',
                  numeric_agg = 'mean', max_k = 10, xtr_outlier_rem = True)
MAX_K = PARAMETERS['max_k']

SAVE_FIGURES = True
FIG_DIR, RES_DIR = Path('figures_sepsis'), Path('results_sepsis')   # kept apart from the RTFM notebook's figures/ and results/
FIG_DPI_EXPORT = 120
FIG_DIR.mkdir(exist_ok = True); RES_DIR.mkdir(exist_ok = True)

DyLoPro draws every figure with `plt.show()`. The helper below saves each figure to `figures_sepsis/` just before it is shown, under the name passed to `fig(...)`. `save_table(...)` writes a table to `results_sepsis/` as CSV and displays it.

In [ ]:
_next_fig = {'name': None}
_original_show = plt.show

def fig(name):
    """Name the next DyLoPro figure; it is saved as figures_sepsis/<name>.png when shown."""
    _next_fig['name'] = name

def _show_and_save(*args, **kwargs):
    if SAVE_FIGURES and _next_fig['name']:
        for i, num in enumerate(plt.get_fignums()):
            suffix = '' if i == 0 else f'_{i}'
            plt.figure(num).savefig(FIG_DIR / f"{_next_fig['name']}{suffix}.png", dpi = FIG_DPI_EXPORT, bbox_inches = 'tight')
        _next_fig['name'] = None
    return _original_show(*args, **kwargs)

plt.show = _show_and_save

def save_table(df, name, index = True):
    df.to_csv(RES_DIR / f'{name}.csv', index = index)
    return df

Reading in the event log, recording its SHA-256 checksum, and preprocessing it (see `preprocess_sepsis()` in `preprocess_case_studies.py`):
- __Case features__: the patient attributes (`Age`, `Diagnose` and 22 boolean SIRS-criteria, diagnostic and clinical flags such as `Hypotensie`, `Oligurie` or `DisfuncOrg`) are only recorded on one event per case (mostly 'ER Registration'). They are lifted to the case level as `'case:<attribute>'`. Missing diagnoses are labeled `'Unknown'`.
- __Binary case outcomes__: the Sepsis Cases event log has no outcome column. Following the outcome-oriented predictive process monitoring literature ([Teinemaa et al.](https://doi.org/10.1145/3301300)), we construct two clinically meaningful binary outcomes:
    - `'case admitted IC'`: the patient is admitted to intensive care ('Admission IC') at some point. (10.5% of the cases)
    - `'case returned ER'`: the patient returns to the emergency room ('Return ER') after being released. (28.0% of the cases)
- __Categorical case feature__ `'case:release type'`: the type of release ('Release A' - 'Release E'), or 'No release' if the patient was never released from the hospital in the log (e.g. patients that leave after the ER).

In [ ]:
from preprocess_case_studies import preprocess_sepsis
assert DATA_PATH.exists(), f'{DATA_PATH} not found'
DATA_SHA256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
log = pm4py.read_xes(str(DATA_PATH))
print(f'file: {DATA_PATH} ({DATA_PATH.stat().st_size:,} bytes), SHA-256 {DATA_SHA256}')
print(f"{log['case:concept:name'].nunique():,} cases, {len(log):,} events, {log['concept:name'].nunique()} activities")
log = preprocess_sepsis(log)

Specifying the available features.

In [ ]:
clinical_flags = [col for col in log.columns if col.startswith('case:') and log[col].dtype == bool]
categorical_casefeatures = ['case:release type', 'case:Diagnose'] + clinical_flags
numeric_casefeatures = ['case:Age']
categorical_eventfeatures = ['org:group', 'concept:name']
numeric_eventfeatures = ['Leucocytes', 'CRP', 'LacticAcid']

Initializing a __DynamicLogPlots__ instance as __sepsis_plots__. We start off with `'case admitted IC'` as the outcome, and switch between the two outcomes with the `set_outcomeColumn(outcome)` method.

In [ ]:
sepsis_plots = dlp.DynamicLogPlots(log, case_id_key = 'case:concept:name', activity_key = 'concept:name',
                                   timestamp_key = 'time:timestamp', categorical_casefeatures = categorical_casefeatures,
                                   numerical_casefeatures = numeric_casefeatures,
                                   categorical_eventfeatures = categorical_eventfeatures,
                                   numerical_eventfeatures = numeric_eventfeatures, outcome = 'case admitted IC')

All of __DyLoPro__'s plotting methods construct time series by deriving real-valued measures for a chronologically ordered set of sublogs, determined by the `frequency` and `case_assignment` parameters.

In contrast to the BPIC17 and BPIC19 case studies, we use __`frequency='monthly'`__ for all visualizations. With only 1,050 cases spread over 16 months (on average +- 66 new cases per month, or +- 15 per week), weekly sublogs would contain too few cases to yield representative aggregations. With `case_assignment='first_event'`, each sublog consists of all cases (patients) that arrived at the ER in one particular month.

__NOTE__: For the Sepsis plots, we use the default settings for the following common parameters unless specified otherwise:
- time_unit = 'days'
- frequency = 'monthly' _(non-default)_
- case_assignment = 'first_event'
- numeric_agg = 'mean'
- max_k = 10
- xtr_outlier_rem = True

## 0. General observations and remarks

1. Every plotting method also provides a first global overview of the number of cases contained within each monthly interval, together with the mean throughput time (`plt_type='type_tt'`), the mean case length in Number of Events Per Case, aka NEPC (`plt_type='type_events_case'`) or the fraction of positive cases (`plt_type='type_outcome'`).

1. __Number of cases__: between 27 and 109 new cases per month. The first months (2013-11: 34 cases, 2013-12: 44 cases) and the last month (2015-02: 27 cases) contain noticeably fewer cases, and the aggregations for these months should be interpreted with care.

1. __Bias towards the end of the event log (right-censoring of 'Return ER')__: the last case starts on 2015-02-26, but events are recorded until 2015-06-05. Patients that return to the ER do so on average 81 days (median 47 days, up to 417 days) after being released. Patients arriving later in the log therefore had less time to return before the extraction date, and their return is more often not (yet) recorded. This is clearly visible in the outcome time series of `'case returned ER'`: it fluctuates mostly between 28% and 43% until 2014-09, after which it drops to 22% (2014-10, 2014-11), 18% (2014-12) and 13% (2015-01). _(The value of 30% in 2015-02 is based on only 27 cases.)_ This should be taken into account when interpreting any pattern related to `'case returned ER'`.

1. __GENERAL TRENDS__:
    - __Throughput Time (TT)__: the monthly mean TT declines gradually from +- 45-65 days at the start of the log to +- 10 days in 2015. This decline is __not__ a sign of faster treatment. The TT of a case includes the time until the patient's return to the ER: patients that return have a mean TT of 89 days vs. 4.9 days for patients that do not return. The decline in the mean TT is therefore mostly a consequence of the right-censoring bias of 'Return ER' described above. (Visible in e.g. the upper left subplot of [this visualization](#11-representation-type--throughput-time-tt).)
    - __Case Length (NEPC)__: in contrast to the TT, the monthly mean NEPC remains stable at +- 13-17 events per case over the entire log.
    - __Control-flow__: apart from the decline of 'Return ER', the fraction of cases containing each of the most frequent activities remains stable over time. No sudden control-flow drift (like the one in BPIC19 around 2018-09) is observed.
    - __Admission to IC__: the fraction of cases admitted to intensive care fluctuates between 6% and 17% per month without any clear trend.

#### 0.1 Supporting analysis: new cases, throughput time, case length and outcomes per month

In [ ]:
CASE_ID, ACT, TS = 'case:concept:name', 'concept:name', 'time:timestamp'
OUTCOMES = ['case admitted IC', 'case returned ER']
cases = log.groupby(CASE_ID).agg(start = (TS, 'min'), end = (TS, 'max'), n_events = (ACT, 'size'),
                                 **{o: (o, 'first') for o in OUTCOMES})
cases['TT_days'] = (cases['end'] - cases['start']).dt.total_seconds() / 86400
cases['month'] = cases['start'].dt.tz_convert(None).dt.to_period('M')

per_month = cases.groupby('month').agg(cases = ('n_events', 'size'), mean_TT_days = ('TT_days', 'mean'),
                                       mean_NEPC = ('n_events', 'mean'), **{o: (o, 'mean') for o in OUTCOMES})
per_month.index = per_month.index.astype(str)
per_month.loc['all months'] = [len(cases), cases['TT_days'].mean(), cases['n_events'].mean()] + [cases[o].mean() for o in OUTCOMES]
save_table(per_month.round(3).astype({'cases': int}), 'table_0_1_per_month')

#### 0.2 Supporting analysis: right-censoring of 'Return ER'

In [ ]:
print('first case starts:', cases['start'].min(), '| last case starts:', cases['start'].max(), '| last event:', log[TS].max())
returns = log[log[ACT] == 'Return ER'].groupby(CASE_ID)[TS].min()
releases = log[log[ACT].str.startswith('Release')].groupby(CASE_ID)[TS].max()
days_to_return = ((returns - releases.reindex(returns.index)).dt.total_seconds() / 86400).dropna()
print('\nDays from release to return to the ER:'); print(days_to_return.describe().round(1).to_string())
print('\nMean TT (days) of patients that do (1) and do not (0) return to the ER:')
print(cases.groupby('case returned ER')['TT_days'].mean().round(1).to_string())

The helper below summarises, for every level of a feature, how many cases have it, their mean TT and NEPC, and the fraction of positive cases for both outcomes among the cases with and without that level, over the whole log. It backs up the observations under each plot. For event features, a case "has" a level if at least one of its events has it, exactly as in DyLoPro (paper Table 1).

In [ ]:
def level_summary(feature, k = MAX_K, save = True):
    has = log.loc[log[feature].notna(), [CASE_ID, feature]].drop_duplicates()
    m = has.merge(cases, left_on = CASE_ID, right_index = True)
    out = m.groupby(feature).agg(cases = (CASE_ID, 'size'), mean_TT_days = ('TT_days', 'mean'),
                                 mean_NEPC = ('n_events', 'mean'), **{o: (o, 'mean') for o in OUTCOMES})
    out.insert(1, 'share_of_cases', out['cases'] / len(cases))
    for o in OUTCOMES:
        out[o + ' (without)'] = [cases.loc[~cases.index.isin(m.loc[m[feature] == lvl, CASE_ID]), o].mean() for lvl in out.index]
    out = out.sort_values('cases', ascending = False).head(k).round(3)
    return save_table(out, 'levels_' + feature.replace(':', '_')) if save else out

## 1. Variants

The Sepsis Cases event log contains __846 distinct variants for only 1,050 cases__. The most frequent variant covers only 35 cases (3.3%), and the 10 most frequent variants together only 136 cases (13%). The process is therefore highly unstructured, and examining control-flow dynamics by means of individual variants is of limited use. _(The [Directly-Follows-Relations](#2-directly-follows-relations-dfrs) give a better view on the control-flow.)_

- The most frequent variants are almost all short pathways in which the patient is not admitted to the hospital (e.g. variant 1: 'ER Registration', 'ER Triage', 'ER Sepsis Triage'). Only variant 8 contains 'Admission NC'. Consequently, their mean TT is (close to) 0 days, compared to +- 10-65 days for all other cases.
- Since each of these variants only accounts for 0-8% of the cases per month, their monthly fractions are very volatile, and no meaningful trends can be identified.

In [ ]:
var_df = sepsis_plots.get_var_df(max_k = 10, counts = True)
var_df['share'] = (var_df['variant count'] / len(cases)).round(3)
print(f"{log.groupby(CASE_ID)[ACT].apply(tuple).nunique()} distinct variants; top 10 cover {var_df['share'].sum():.1%} of cases")
save_table(var_df, 'table_1_top_variants')

#### 1.1 Representation type : __Throughput Time (TT)__

In [ ]:
fig('01_1_variants_tt')
sepsis_plots.topK_variants_evol(plt_type = 'type_tt', frequency = 'monthly')

## 2. Directly-Follows-Relations (DFRs)

We investigate the dynamics over time for the 10 most frequently occurring DFRs. The following interesting trends and patterns were identified:

1. __Lab tests are recorded in batches__: 6 of the 10 most frequent DFRs are pairs of lab activities ('Leucocytes', 'CRP' or 'LacticAcid'). For 5 of them (DFRs 1, 2, 5, 6 and 10), the two lab results are usually registered at the same timestamp (median [DFR performance](#23-representation-type--dfr-performance) of 0), so their order in the log is arbitrary (91% of all lab events share their timestamp with another lab event of the same case). For example, both __DFR 1__ ('Leucocytes', 'CRP') and __DFR 2__ ('CRP', 'Leucocytes') occur. This partly explains the very high number of variants. The sixth, __DFR 9__ ('Leucocytes', 'Leucocytes'), is different: it is a repeated measurement one day apart (see below).
1. __DFR 9 ('Leucocytes', 'Leucocytes')__: occurs in +- 21% of the cases, on average 2.1 times per case, with a median DFR performance of exactly 1 day. This corresponds to the daily repetition of the leucocytes measurement during a hospital stay. Cases containing DFR 9 have a considerably higher [mean NEPC](#22-representation-type--case-length-nepc) (+- 25 vs. 12 events) and [mean TT](#21-representation-type--throughput-time-tt), and a much higher [fraction of IC admissions](#24-representation-type--outcome-case-admitted-ic) (26% vs. 7%). Repeated lab measurements hence seem to be a marker of more severe cases.
1. __Lab DFRs including 'LacticAcid' (DFRs 5, 6 and 10)__: cases containing these DFRs are admitted to IC far more often (21-27%) than cases that do not (+- 3%). Lactic acid is a marker for sepsis severity, so it is plausible that it is mainly measured for the more severe patients.
1. __ER activities (DFRs 3 and 4)__: 'ER Registration' -> 'ER Triage' -> 'ER Sepsis Triage' occurs exactly once in almost every case (86-93%). Their DFR performance is in the order of minutes, and remains stable over time.
1. __DFR 8 ('IV Antibiotics', 'Admission NC')__: occurs in +- 47% of the cases. Cases containing DFR 8 have a lower fraction of IC admissions (5% vs. 15%): these are the patients that are admitted to a normal care ward right after their antibiotics.
1. __Over time__: the fraction of cases containing each DFR and the mean number of occurrences per case fluctuate, but no consistent trends or sudden changes are observed. The DFR performance time series are volatile due to the small number of cases per month.

In [ ]:
dfr_df = sepsis_plots.get_DFR_df(max_k = 10, counts = True)
print(dfr_df.to_string())

In [ ]:
lab = log[log[ACT].isin(['Leucocytes', 'CRP', 'LacticAcid'])]
print(f"{lab.duplicated([CASE_ID, TS], keep = False).mean():.0%} of all lab events share their timestamp with another lab event of the same case")

dfrs = log[[CASE_ID, ACT, TS]].copy()
dfrs['next_act'] = dfrs.groupby(CASE_ID)[ACT].shift(-1)
dfrs['perf_days'] = (dfrs.groupby(CASE_ID)[TS].shift(-1) - dfrs[TS]).dt.total_seconds() / 86400
dfrs = dfrs.dropna(subset = ['next_act'])
rows = []
for k, (a, b) in enumerate(dfr_df['DFR'], 1):
    sub = dfrs[(dfrs[ACT] == a) & (dfrs['next_act'] == b)]
    per_case = sub.groupby(CASE_ID).size()
    has = cases.index.isin(per_case.index)
    rows.append({'DFR number': k, 'DFR': (a, b), 'fraction cases': has.mean(), 'occurrences per case (with)': per_case.mean(),
                 'median performance (days)': sub['perf_days'].median(),
                 'NEPC with': cases.loc[has, 'n_events'].mean(), 'NEPC without': cases.loc[~has, 'n_events'].mean(),
                 'IC with': cases.loc[has, 'case admitted IC'].mean(), 'IC without': cases.loc[~has, 'case admitted IC'].mean()})
save_table(pd.DataFrame(rows).set_index('DFR number').round(3), 'table_2_dfr_summary')

#### 2.1 Representation type : __Throughput Time (TT)__

In [ ]:
fig('02_1_dfr_tt')
sepsis_plots.topK_dfr_evol(plt_type = 'type_tt', frequency = 'monthly')

__Note on the left-hand plots__: for DFR 3, 4, 7 and 8 you only see the orange line, because the blue line lies exactly underneath it. These DFRs never occur more than once in a case, and then both lines give the same value.

__Example__: a month with 10 cases, in which 9 cases contain the DFR once and 1 case does not:

blue (mean number of occurrences per case) = (1+1+1+1+1+1+1+1+1+0) / 10 = __0.9__
orange (fraction of cases with at least one occurrence) = 9 / 10 = __0.9__
The lines only differ when a case contains the DFR more than once. If one case contained it twice, blue would be (2+1+1+1+1+1+1+1+1+0) / 10 = 1.0, while orange stays 0.9. This happens for e.g. DFR 1 ('Leucocytes' -> 'CRP').

Look below to see how they overlap. 

In [ ]:
import matplotlib.pyplot as plt

def show_with_visible_overlaps(plot_call):
    """Run a DyLoPro plotting call and draw the 'Fraction cases' lines dashed and
    semi-transparent, so that lines lying exactly underneath them stay visible."""
    original_show = plt.show
    plt.show = lambda *args, **kwargs: None  # stop DyLoPro from displaying the figure itself
    try:
        plot_call()
    finally:
        plt.show = original_show
    fig = plt.gcf()
    for ax in fig.axes:
        for line in ax.get_lines():
            if line.get_label().startswith('Fraction cases'):
                line.set_linestyle('--')
                line.set_linewidth(3)
                line.set_alpha(0.4)
    plt.show()


In [ ]:
fig('02_1_dfr_tt_overlaps')
show_with_visible_overlaps(lambda: sepsis_plots.topK_dfr_evol(plt_type = 'type_tt', frequency = 'monthly'))

#### 2.2 Representation type : __Case Length (NEPC)__

In [ ]:
fig('02_2_dfr_nepc')
sepsis_plots.topK_dfr_evol(plt_type = 'type_events_case', frequency = 'monthly')

#### 2.3 Representation type : __DFR Performance__

In [ ]:
fig('02_3_dfr_performance')
sepsis_plots.topK_dfr_evol(plt_type = 'type_dfr_performance', frequency = 'monthly')

#### 2.4 Representation type : __Outcome__ 'case admitted IC'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case admitted IC')
fig('02_4_dfr_outcome_IC')
sepsis_plots.topK_dfr_evol(plt_type = 'type_outcome', frequency = 'monthly')

## 3. Categorical Case Features

Only the categorical case features for which interesting trends and patterns over time were identified are discussed here.

### 3.1 Categorical Case Feature: 'case:release type'

1. __Fraction of cases__: 'Release A' is by far the most common release type (64% of the cases, stable between 58-71% per month), followed by 'No release' (26%), 'Release B' (5%), 'Release C' (2%), 'Release D' (2%) and 'Release E' (<1%). These fractions remain relatively stable over time.
1. [__Mean TT__](#311-caserelease-type---representation-type-throughput-time-tt):
    - __'No release'__: TT of (almost) 0 days. These are the patients that leave after the ER and are never admitted.
    - __'Release A'__: consistently and considerably higher mean TT than other cases. The mean TT of 'Release A'-patients declines from +- 65-100 days to +- 15 days over time. 41% of the 'Release A'-patients return to the ER, so this decline again reflects the right-censoring of 'Return ER'.
    - __'Release B'__: mean TT consistently low (+- 5-10 days), although these patients are admitted to the hospital. None of the 56 'Release B'-patients returns to the ER (see below).
1. [__Mean NEPC__](#312-caserelease-type---representation-type-case-length-nepc): patients with 'No release' have short cases (+- 7 events), compared to +- 16 events for 'Release A' and +- 19-24 events for the less frequent release types B-E.
1. __Outcome__:
    - [__'case admitted IC'__](#313a-outcome---case-admitted-ic): patients released with 'Release B' (23%), 'Release D' (21%) and 'Release E' (33%) were more often admitted to intensive care than 'Release A'-patients (13%). Patients with 'No release' are (almost) never admitted to IC (1%).
    - [__'case returned ER'__](#313b-outcome---case-returned-er): __only patients released with 'Release A', 'C', 'D' or 'E' return to the ER__. Not a single 'Release B'-patient returns. This suggests that 'Release B' might correspond to a release type after which returning to the ER is not possible (e.g. transfer to another institution, or deceased).

In [ ]:
level_summary('case:release type')

#### 3.1.1 'case:release type' - Representation type: __Throughput Time (TT)__

##### The left panel shows how common it is each month, and the right panel shows how long those patients' cases take.

In [ ]:
fig('03_1_1_releasetype_tt')
sepsis_plots.topK_categorical_caseftr_evol(case_feature = 'case:release type', plt_type = 'type_tt', frequency = 'monthly')

#### 3.1.2 'case:release type' - Representation type: __Case Length (NEPC)__

In [ ]:
fig('03_1_2_releasetype_nepc')
sepsis_plots.topK_categorical_caseftr_evol(case_feature = 'case:release type', plt_type = 'type_events_case', frequency = 'monthly')

#### 3.1.3 'case:release type' - Representation type: __Outcome__

##### 3.1.3.A Outcome - 'case admitted IC'

The gaps (places where no point is shown) aren't missing data. In those months zero patients in that group went to intensive care, and DyLoPro draws a 0 as a gap instead of a point at 0.

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case admitted IC')
fig('03_1_3_releasetype_outcome_A_IC')
sepsis_plots.topK_categorical_caseftr_evol(case_feature = 'case:release type', plt_type = 'type_outcome', frequency = 'monthly')

##### 3.1.3.B Outcome - 'case returned ER'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case returned ER')
fig('03_1_3_releasetype_outcome_B_returnedER')
sepsis_plots.topK_categorical_caseftr_evol(case_feature = 'case:release type', plt_type = 'type_outcome', frequency = 'monthly')

### 3.2 Categorical Case Feature: 'case:Diagnose'

Categorical case feature __'case:Diagnose'__ has 147 distinct levels (anonymized diagnosis codes, incl. 'Unknown'). We investigate the 10 most frequently occurring levels.

1. __'Unknown'__ (24% of the cases): these are the patients without a recorded diagnosis. They are (almost) never admitted to IC (1%), never return to the ER, and have a TT of (almost) 0 days. These largely coincide with the 'No release' patients that leave after the ER.
1. [__Outcome 'case admitted IC'__](#321a-outcome---case-admitted-ic): __all 44 patients with diagnosis 'G' are admitted to intensive care__, in every single month, while only 7% of all other patients are. This pattern can be regarded as __data leakage__: the diagnosis code 'G' seems to be registered as a consequence of (or together with) the IC admission. This deserves attention when e.g. predicting IC admission based on the patient attributes.
1. [__Outcome 'case returned ER'__](#321b-outcome---case-returned-er): patients with diagnoses 'D' (61%), 'H' (47%) and 'B' (43%) return to the ER more often than average (28%), although the monthly time series are volatile due to the small number of patients per diagnosis.

In [ ]:
print(f"{log['case:Diagnose'].nunique()} distinct levels")
level_summary('case:Diagnose')

#### 3.2.1 'case:Diagnose' - Representation type: __Outcome__

##### 3.2.1.A Outcome - 'case admitted IC'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case admitted IC')
fig('03_2_diagnose_outcome_A_IC')
sepsis_plots.topK_categorical_caseftr_evol(case_feature = 'case:Diagnose', plt_type = 'type_outcome', frequency = 'monthly')

##### 3.2.1.B Outcome - 'case returned ER'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case returned ER')
fig('03_2_diagnose_outcome_B_returnedER')
sepsis_plots.topK_categorical_caseftr_evol(case_feature = 'case:Diagnose', plt_type = 'type_outcome', frequency = 'monthly')

### 3.3 Categorical Case Features: clinical flags

The 22 boolean clinical flags fall into two groups:

1. __Rare severity indicators__ - `'case:Hypotensie'` (54 patients), `'case:Oligurie'` (25), `'case:DisfuncOrg'` (69, organ dysfunction) and `'case:Hypoxie'` (20): patients with these flags are admitted to IC far more often. E.g. 56% of the patients with hypotension or oliguria and 45% of the patients with organ dysfunction are admitted to IC, vs. +- 8-9% of the other patients. This difference is consistent over time, although the monthly time series for the 'True' level are volatile due to the small number of patients. [(Example: 'case:Hypotensie')](#331-casehypotensie---representation-type-outcome-case-admitted-ic)
1. __Common indicators__ - e.g. `'case:InfectionSuspected'` (81% True), `'case:SIRSCriteria2OrMore'` (81%), `'case:DiagnosticIC'` (81%), `'case:Infusion'` (76%): patients with these flags set to 'True' have a consistently and considerably higher mean TT (e.g. 31 vs. 17 days for `'case:InfectionSuspected'`) and NEPC (+- 16 vs. 8 events). [(Example: 'case:InfectionSuspected')](#332-caseinfectionsuspected---representation-type-throughput-time-tt)

The fractions of the 'True' and 'False' levels of all flags remain stable over time.

In [ ]:
flag_summary = pd.DataFrame({f: level_summary(f, save = False).loc[True] for f in clinical_flags}).T
save_table(flag_summary.sort_values('cases'), 'levels_clinical_flags_true')

#### 3.3.1 'case:Hypotensie' - Representation type: __Outcome__ 'case admitted IC'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case admitted IC')
fig('03_3_1_hypotensie_outcome_IC')
sepsis_plots.topK_categorical_caseftr_evol(case_feature = 'case:Hypotensie', plt_type = 'type_outcome', frequency = 'monthly')

#### 3.3.2 'case:InfectionSuspected' - Representation type: __Throughput Time (TT)__

In [ ]:
fig('03_3_2_infectionsuspected_tt')
sepsis_plots.topK_categorical_caseftr_evol(case_feature = 'case:InfectionSuspected', plt_type = 'type_tt', frequency = 'monthly')

## 4. Numerical Case Features

The Sepsis Cases event log has only 1 numerical case feature, `'case:Age'` (rounded to 5-year intervals, between 20 and 90).

1. __Mean value__: the monthly mean age of the patients remains stable at +- 66-76 years, without a clear trend.
1. [__Mean ratio TT / unit 'case:Age'__](#41-caseage---representation-type-throughput-time-tt): follows the evolution of the overall mean TT, and hence declines over time (see [Section 0](#0-general-observations-and-remarks)).
1. __Outcome__:
    - [__'case admitted IC'__](#43a-outcome---case-admitted-ic): no consistent difference in mean age between patients that are and are not admitted to IC (overall mean 67.4 vs. 70.4 years).
    - [__'case returned ER'__](#43b-outcome---case-returned-er): in 13 of the 16 months, patients that return to the ER are on average older than patients that do not (overall mean 73.3 vs. 68.8 years).

In [ ]:
age = cases.join(log.drop_duplicates(CASE_ID).set_index(CASE_ID)['case:Age'])
save_table(pd.DataFrame({o: age.groupby(o)['case:Age'].mean() for o in OUTCOMES}).round(1), 'table_4_mean_age_per_outcome')

### 4.1 'case:Age' - Representation type: __Throughput Time (TT)__

Plot 4: For each patient, DyLoPro computes throughput time in days ÷ age in years, then averages this per month

In [ ]:
fig('04_1_age_tt')
sepsis_plots.num_casefts_evol(numeric_case_list = numeric_casefeatures, plt_type = 'type_tt', frequency = 'monthly')

### 4.2 'case:Age' - Representation type: __Case Length (NEPC)__

In [ ]:
fig('04_2_age_nepc')
sepsis_plots.num_casefts_evol(numeric_case_list = numeric_casefeatures, plt_type = 'type_events_case', frequency = 'monthly')

### 4.3 'case:Age' - Representation type: __Outcome__

#### 4.3.A Outcome - 'case admitted IC'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case admitted IC')
fig('04_3_age_outcome_A_IC')
sepsis_plots.num_casefts_evol(numeric_case_list = numeric_casefeatures, plt_type = 'type_outcome', frequency = 'monthly')

#### 4.3.B Outcome - 'case returned ER'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case returned ER')
fig('04_3_age_outcome_B_returnedER')
sepsis_plots.num_casefts_evol(numeric_case_list = numeric_casefeatures, plt_type = 'type_outcome', frequency = 'monthly')

## 5. Categorical Event Features

### 5.1 Categorical Event Feature: 'org:group'

**'org:group'** contains the (anonymized) department or group that executes an event. The Sepsis Cases event log contains 26 distinct groups. Several groups are tied to specific activities:
- __'A'__ and __'L'__ register patients at the ER ('ER Registration') and administer 'IV Antibiotics'.
- __'B'__ executes all lab tests ('Leucocytes', 'CRP', 'LacticAcid').
- __'C'__ is involved in every single case (ER triage).
- __'E'__ executes all release events ('Release A' - 'Release E').
- __'?'__ executes all 'Return ER' events, and only those.
- __'F'__, __'O'__, __'G'__, ... are the normal care wards ('Admission NC').

The following patterns were observed:
1. __Fraction of cases__: stable over time for all groups, except for group __'?'__, whose fraction follows the decline of 'Return ER' (see [Section 0](#0-general-observations-and-remarks)), and group __'L'__ (ER registration), which is involved in 1-11% of the cases per month.
1. [__Outcome 'case returned ER'__](#513b-outcome---case-returned-er): the involvement of group __'?'__ is identical to the outcome 'case returned ER' (100% vs. 0%) in every month. This is __data leakage__ by construction: group '?' only executes 'Return ER'. Similarly, cases without group 'B' (lab) or 'E' (Release A) never return to the ER.
1. [__Outcome 'case admitted IC'__](#513a-outcome---case-admitted-ic): cases in which the normal care wards __'F'__ (16.5%), __'O'__ (14.5%) and __'I'__ (13.6%) are involved are admitted to IC slightly more often than other cases (+- 9-10%).
1. [__Mean TT__](#511-orggroup---representation-type-throughput-time-tt): cases involving group '?' have by far the highest TT (89 vs. 5 days), again because their TT includes the period until the patient's return.

In [ ]:
print(log.groupby('org:group')[ACT].agg(lambda s: s.value_counts().to_dict()).to_string())
level_summary('org:group')

#### 5.1.1 'org:group' - Representation type: __Throughput Time (TT)__

In [ ]:
fig('05_1_1_group_tt')
sepsis_plots.topK_categorical_eventftr_evol(event_feature = 'org:group', plt_type = 'type_tt', frequency = 'monthly')

#### 5.1.2 'org:group' - Representation type: __Case Length (NEPC)__

In [ ]:
fig('05_1_2_group_nepc')
sepsis_plots.topK_categorical_eventftr_evol(event_feature = 'org:group', plt_type = 'type_events_case', frequency = 'monthly')

#### 5.1.3 'org:group' - Representation type: __Outcome__

##### 5.1.3.A Outcome - 'case admitted IC'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case admitted IC')
fig('05_1_3_group_outcome_A_IC')
sepsis_plots.topK_categorical_eventftr_evol(event_feature = 'org:group', plt_type = 'type_outcome', frequency = 'monthly')

##### 5.1.3.B Outcome - 'case returned ER'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case returned ER')
fig('05_1_3_group_outcome_B_returnedER')
sepsis_plots.topK_categorical_eventftr_evol(event_feature = 'org:group', plt_type = 'type_outcome', frequency = 'monthly')

### 5.2 Categorical Event Feature: 'concept:name'

**'concept:name'** contains the activity labels. The Sepsis Cases event log contains 16 distinct activities. We investigate the 10 most frequently occurring activities.

1. __Always present__: 'ER Registration', 'ER Triage' and 'ER Sepsis Triage' occur in (almost) every case, in every month.
1. __Lab tests__: 'Leucocytes' and 'CRP' occur in 92-100% of the cases per month. 'LacticAcid' is measured in +- 91% of the cases in the first two months, after which it drops to +- 75-85% for the rest of the log.
1. __'Admission NC'__: occurs in 68-84% of the cases per month. All patients that return to the ER were admitted to a normal care ward first: __the fraction of returning patients among cases without 'Admission NC' is 0 in every month__ ([outcome 'case returned ER'](#523-conceptname---representation-type-outcome-case-returned-er)).
1. __'IV Antibiotics' and 'IV Liquid'__: patients that receive IV antibiotics return to the ER more often (32% vs. 14%) and are admitted to IC more often (12% vs. 4%) than patients that do not.
1. __'Return ER'__: declines from +- 30-43% of the cases to 13-22% towards the end of the log, due to the right-censoring discussed in [Section 0](#0-general-observations-and-remarks).
1. [__Mean TT__](#521-conceptname---representation-type-throughput-time-tt) & [__Mean NEPC__](#522-conceptname---representation-type-case-length-nepc): the presence of 'Admission NC', 'IV Antibiotics' and the lab activities goes together with a considerably higher TT and NEPC, consistently over time.

In [ ]:
level_summary('concept:name')

#### 5.2.1 'concept:name' - Representation type: __Throughput Time (TT)__

In [ ]:
fig('05_2_1_activity_tt')
sepsis_plots.topK_categorical_eventftr_evol(event_feature = 'concept:name', plt_type = 'type_tt', frequency = 'monthly')

#### 5.2.2 'concept:name' - Representation type: __Case Length (NEPC)__

In [ ]:
fig('05_2_2_activity_nepc')
sepsis_plots.topK_categorical_eventftr_evol(event_feature = 'concept:name', plt_type = 'type_events_case', frequency = 'monthly')

#### 5.2.3 'concept:name' - Representation type: __Outcome__ 'case returned ER'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case returned ER')
fig('05_2_3_activity_outcome_returnedER')
sepsis_plots.topK_categorical_eventftr_evol(event_feature = 'concept:name', plt_type = 'type_outcome', frequency = 'monthly')

## 6. Numerical Event Features

The Sepsis Cases event log contains 3 numerical event features, the lab values `'Leucocytes'`, `'CRP'` and `'LacticAcid'`, each recorded on the event of the corresponding lab activity. Since these lab tests can be repeated multiple times during a patient's stay, we project each case's sequence of values to the case level by taking the __first__ measured value (`numEventFt_transform='first'`), i.e. the value measured at (or shortly after) the patient's arrival at the ER. This way, the values are measured before the outcome occurs.

1. __Mean values__: the monthly mean values of the 3 lab values fluctuate without a clear trend (Leucocytes +- 12-15, apart from 22.1 in the first month 2013-11, CRP +- 85-135, LacticAcid +- 1.8-2.4).
1. [__Outcome 'case admitted IC'__](#63a-outcome---case-admitted-ic):
    - `'LacticAcid'`: the first lactic acid value of patients that are admitted to IC is almost consistently higher than that of other patients (overall mean 2.9 vs. 1.9). An elevated lactate is a known indicator of severe sepsis.
    - `'CRP'`: in most months, the mean first CRP value of patients admitted to IC is clearly higher (overall mean 148 vs. 105), but this difference is not consistent over time.
    - `'Leucocytes'`: no consistent difference (15.0 vs. 13.8).
1. [__Outcome 'case returned ER'__](#63b-outcome---case-returned-er): no consistent differences in the first lab values between patients that do and do not return to the ER.

In [ ]:
first_values = pd.DataFrame({f: log.dropna(subset = [f]).groupby(CASE_ID)[f].first() for f in numeric_eventfeatures})
first_values = first_values.join(cases[OUTCOMES + ['month']])
print('Monthly mean of the first measured value:')
print(first_values.groupby('month')[numeric_eventfeatures].mean().round(2).T.to_string())
save_table(pd.concat({o: first_values.groupby(o)[numeric_eventfeatures].mean() for o in OUTCOMES}).round(2), 'table_6_first_lab_value_per_outcome')

### 6.1 Numerical Event Features - Representation type: __Throughput Time (TT)__

In [ ]:
fig('06_1_lab_first_tt')
sepsis_plots.num_eventfts_evol(numeric_event_list = numeric_eventfeatures, plt_type = 'type_tt', numEventFt_transform = 'first', frequency = 'monthly')

### 6.2 Numerical Event Features - Representation type: __Case Length (NEPC)__

In [ ]:
fig('06_2_lab_first_nepc')
sepsis_plots.num_eventfts_evol(numeric_event_list = numeric_eventfeatures, plt_type = 'type_events_case', numEventFt_transform = 'first', frequency = 'monthly')

### 6.3 Numerical Event Features - Representation type: __Outcome__

#### 6.3.A Outcome - 'case admitted IC'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case admitted IC')
fig('06_3_lab_first_outcome_A_IC')
sepsis_plots.num_eventfts_evol(numeric_event_list = numeric_eventfeatures, plt_type = 'type_outcome', numEventFt_transform = 'first', frequency = 'monthly')

#### 6.3.B Outcome - 'case returned ER'

In [ ]:
sepsis_plots.set_outcomeColumn(outcome = 'case returned ER')
fig('06_3_lab_first_outcome_B_returnedER')
sepsis_plots.num_eventfts_evol(numeric_event_list = numeric_eventfeatures, plt_type = 'type_outcome', numEventFt_transform = 'first', frequency = 'monthly')

## Export manifest
Everything needed to cite this run in the report's reproducibility paragraph.

In [ ]:
manifest = {**ENV, 'data file': str(DATA_PATH), 'data DOI': DATA_DOI, 'download date': DOWNLOAD_DATE,
            'data SHA-256': DATA_SHA256, 'original code': ORIGINAL_CODE, 'parameters': PARAMETERS,
            'runs': 1, 'seeds': 'none (DyLoPro is deterministic)'}
(RES_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent = 2))
print(json.dumps(manifest, indent = 2))
print(f"\n{len(list(FIG_DIR.glob('*.png')))} figures in {FIG_DIR}/, {len(list(RES_DIR.glob('*.csv')))} tables in {RES_DIR}/")